In [1]:
!pip install pm4py -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
mimicel_path = "/content/drive/MyDrive/PM_assessments_Logs/PALSYN/PALSYN/data/MIMICEL/mimicel_train.xes"

In [ ]:
import pm4py

event_log = pm4py.read_xes(mimicel_path)

# 데이터 기본 확인
df = pm4py.convert_to_dataframe(event_log)
print(f"총 케이스 수: {df['case:concept:name'].nunique()}")
print(f"총 이벤트 수: {len(df)}")
print(f"액티비티 종류: {df['concept:name'].nunique()}")
print(df.head())

In [ ]:
from PALSYN.synthesizer import LSTMSynthesizer

palsyn_model = LSTMSynthesizer(
    pre_processing={
        "max_clusters": 15,       # 수치 속성 클러스터 수
        "trace_quantile": 0.9,    # 긴 trace 잘라내는 분위수
        "seed": 42,
    },
    model={
        "embedding_output_dims": 128,
        "epochs": 10,             # 논문 재현 시 원래 값으로 맞출 것
        "batch_size": 128,
        "validation_split": 0.15,
        "units_per_layer": [32, 16],
        "dropout": 0.0,
        "bidirectional": True,
    },
    dp_optimizer={
        "epsilon": 15.0,          # 차분 프라이버시 엡실론 (논문 값)
        "learning_rate": 5e-4,
        "l2_norm_clip": 1.0,
    },
)

palsyn_model.fit(event_log)
palsyn_model.save_model("/content/PALSYN/models/mimicel_run")

In [ ]:
from PALSYN.postprocessing.log_postprocessing import clean_xes_file

# 원본 케이스 수와 비슷하게 설정
sample_size = df['case:concept:name'].nunique()

palsyn_model = LSTMSynthesizer.load("/content/PALSYN/models/mimicel_run")
synthetic_log = palsyn_model.sample(sample_size=sample_size, batch_size=100)
synthetic_log_xes = pm4py.convert_to_event_log(synthetic_log)

# 저장
pm4py.write_xes(synthetic_log_xes, "mimicel_synthetic.xes")
clean_xes_file("mimicel_synthetic.xes", "mimicel_synthetic.xes")

# Excel도 저장
synthetic_df = pm4py.convert_to_dataframe(synthetic_log_xes)
synthetic_df["time:timestamp"] = synthetic_df["time:timestamp"].astype(str)
synthetic_df.to_excel("mimicel_synthetic.xlsx", index=False)